In [1]:
import gzip
import torch
import numpy as np
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from torch.nn import init
import argparse

## EDA on SMILES strings

In [7]:
# Peek at randomize file
path="/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/data/pubchem_randomized.smi.gz"
with gzip.open(path, 'rt') as f:
    lines = [f.readline().rstrip() for _ in range(10)]
print(f"\n=== Read the first 10 lines ===")
for l in lines:
    print(l)


=== Read the first 10 lines ===
C[N+](CC(CC(=O)[O-])OC(=O)C)(C)C
C(=O)(OC(CC(=O)O)C[N+](C)(C)C)C
C1(O)C(C(=CC=C1)C(=O)O)O
OC(C)CN
P(=O)(O)(OCC(=O)CN)O
C1N=C(C2N=CN(CC)C=2N=1)N
OC(C(O)(C)CC)C(=O)O
O(P(=O)(O)O)C1C(O)C(O)C(C(O)C1O)O
C12=C(NCC(CN(C3=CC=C(C=C3)C(=O)NC(C(=O)O)CCC(=O)O)C=O)N2)NC(=NC1=O)N
OC1C=C(O)C=C(O)C=1O


In [9]:
# Check total number of strings
with gzip.open(path, 'rt') as f:
    total = sum(1 for _ in f)
print(f"Total lines: {total}")

Total lines: 62129800


In [16]:
## Record the length && unique char-set of the first 100k sample
lengths = []
count = 0
chars = set()

with gzip.open(path, 'rt') as f:
    for line in f:
        l = line.rstrip()
        lengths.append(len(l))
        chars.update(line.rstrip())
        count += 1
        if count >= 100_000:
            break

lengths = np.array(lengths)
print(f"Sample size: {count}")
print(f"Length stats: min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}, median={np.median(lengths):.1f}")
print(f"<50 chars : {(lengths<50).mean():.2f}")
print(f"<100 chars: {(lengths<100).mean():.2f}")
print(f"\nUnique chars in sample: {sorted(chars)}")
print(f"Number of unique chars: {len(chars)}")

Sample size: 100000
Length stats: min=1, max=149, mean=38.2, median=35.0
<50 chars : 0.78
<100 chars: 0.99

Unique chars in sample: ['#', '(', ')', '+', '-', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', ']']
Number of unique chars: 26


## Debug v2 to v3 eval timeout

In [3]:
import torch, time

model = torch.jit.load('/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/code/run4_longMask_v3_genEval_default/debug_v2_to_v3/vae_decoder_len50_Mol50k_smallerVAE.pth')

for i in range(5):
    z = torch.normal(0, 1, size=(1, 1024),device='cuda')
    start = time.time()
    _, out = model(z)
    torch.cuda.synchronize()
    print(f'Call {i}: {time.time()-start:.2f}s, shape={out.shape}, tokens={out[0][:10].tolist()}', flush=True)

print('ALL DONE', flush=True)

Call 0: 0.28s, shape=torch.Size([1, 50]), tokens=[5, 2, 12, 2, 2, 2, 3, 4, 10, 2]
Call 1: 0.13s, shape=torch.Size([1, 50]), tokens=[2, 3, 2, 2, 6, 3, 2, 2, 2, 6]
Call 2: 0.05s, shape=torch.Size([1, 50]), tokens=[2, 2, 2, 12, 3, 5, 2, 2, 12, 4]
Call 3: 0.01s, shape=torch.Size([1, 50]), tokens=[2, 3, 10, 6, 5, 2, 3, 4, 5, 6]
Call 4: 0.01s, shape=torch.Size([1, 50]), tokens=[2, 12, 3, 5, 6, 2, 2, 3, 4, 2]
ALL DONE


In [5]:
class Lang:
    def __init__(self):
        self.chartoindex = {'SOS': 1, 'EOS': 0, 'C': 2, '(': 3,
                '=': 4, 'O': 5, ')': 6, '[': 7, '-': 8, ']': 9,
                'N': 10, '+': 11, '1': 12, 'P': 13, '2': 14,'3': 15,
                '4': 16, 'S': 17, '#': 18, '5': 19,'6': 20, '7': 21,
                'H': 22, 'I': 23, 'B': 24, 'F': 25, '8': 26, '9': 27
                } 
        self.indextochar = {1: 'SOS', 0: 'EOS', 2: 'C', 3: '(',
                4: '=', 5: 'O', 6: ')', 7: '[', 8: '-', 9: ']',
                10: 'N', 11: '+', 12: '1', 13: 'P', 14: '2', 15: '3',
                16: '4', 17: 'S', 18: '#', 19: '5', 20: '6', 21: '7',
                22: 'H', 23: 'I', 24: 'B', 25: 'F', 26: '8', 27: '9'
                }
        self.nchars = 28
        
    def indexesFromSMILES(self, smiles_str):
        index_list = [self.chartoindex[char] for char in smiles_str]
        index_list.append(self.chartoindex["EOS"])
        return np.array(index_list, dtype=np.uint8)
        
    def indexToSmiles(self,indices):
        '''convert list of indices into a smiles string'''
        #todo: be nice and ignore after first EOS before this check
        if 1 in indices:
            print("SOS character encountered in smile")
            return 'x'
        smiles_str = ''.join(list(map(lambda x: self.indextochar[int(x)] if x != 0.0 else 'E',indices)))
        return smiles_str.split('E')[0] #Only want values before output 'EOS' token        


def smilesToStatistics(list_of_smiles,trainsmi):
    '''Return number valid smiles, number of unique molecules, and average number of rings'''
    count_molecules = 0
    cannonical_smiles = set()
    ringcnt = 0
    novelcnt = 0
    for smiles in list_of_smiles:
        if smiles == '':
            continue
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                count_molecules += 1
                can = Chem.MolToSmiles(mol)
                if can not in cannonical_smiles:
                    #print(can)
                    cannonical_smiles.add(can)
                    r = mol.GetRingInfo()
                    ringcnt += r.NumRings()
                    if can not in trainsmi:
                        novelcnt += 1
                    
        except:
            continue
    N = len(cannonical_smiles)
    ringave = 0 if N == 0 else ringcnt/N
    return count_molecules, N, novelcnt, ringave

############################################################
# import torch, time

# model = torch.jit.load('/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/code/run4_longMask_v3_genEval_default/debug_v2_to_v3/vae_decoder_len50_Mol50k_smallerVAE.pth')

# lang = Lang()
# with torch.no_grad():
#     for i in range(5):
#         z = torch.normal(0, 1, size=(1, 1024),device='cuda')
#         start = time.time()
#         _, out = model(z)
#         torch.cuda.synchronize()
#         print(f'Call {i}: {time.time()-start:.2f}s, shape={out.shape}, tokens={out[0][:10].tolist()}', flush=True)
#         zsmile = lang.indexToSmiles(out[0])
#         print(f'zsmile: {zsmile}', flush=True)

# print('ALL DONE', flush=True)



LATENT_DIM = 1024
eval_quant = 2

lang = Lang()
print('about to load')
model = torch.jit.load('/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/code/run4_longMask_v3_genEval_default/debug_v2_to_v3/vae_decoder_len50_Mol50k_smallerVAE.pth')
print('loaded',flush=True)
torch.manual_seed(0)
trainsmi = set()
# if args.train_pickle:
#     trainsmi = pickle.load(open(args.train_pickle,'rb'))
print("read pickle",flush=True)
start_time = time.time() 

generated_smiles = []
# with torch.no_grad():
print('Creating z...', flush=True)
z = torch.zeros((1,LATENT_DIM),device='cuda')
print('z created on cuda -> Running model...', flush=True)
_, output_smile = model(z)
torch.cuda.synchronize() #show true GPU comp. time
print(f'Done in {time.time()-start_time:.1f}s', flush=True)
print(f'Output shape: {output_smile.shape}', flush=True)

print('Converting zsmile...', flush=True)
zsmile = lang.indexToSmiles(output_smile[0])
print(f'zsmile: {zsmile}', flush=True)

print('Starting loop...', flush=True)
for i in range(eval_quant):
    print(f'Iter {i}: creating z...', flush=True)
    z_1 = torch.normal(0, 1, size=(1, LATENT_DIM),device='cuda')
    print(z_1)
    print(f'Iter {i}: running model...', flush=True)
    _, output_smile2 = model(z_1)
    torch.cuda.synchronize()
    print(f'Iter {i}: converting...', flush=True)
    generated_smiles.append(lang.indexToSmiles(output_smile2[0]))
    print(i,flush=True)
    
duration = (time.time()-start_time)
valid_smiles, unique_mols, novelcnt, ringave = smilesToStatistics(generated_smiles,trainsmi)


print("GenerateTime",duration)
print("UniqueSmiles", len(set(generated_smiles)))
print("ValidSmiles", valid_smiles)
print("UniqueAndValidMols", unique_mols)
print("NovelMols", novelcnt)
print("AverageRings", ringave)
print("SMILE",zsmile)



about to load
loaded
read pickle
Creating z...
z created on cuda -> Running model...
Done in 0.1s
Output shape: torch.Size([1, 50])
Converting zsmile...
zsmile: C1C(=CC2C(CC(=C1)CN1CCCC1)(C1CCOC1=CC2C(C=C1N1CC2)
Starting loop...
Iter 0: creating z...
tensor([[ 1.9453, -0.0220,  0.3444,  ..., -0.4102,  1.7316,  0.9163]],
       device='cuda:0')
Iter 0: running model...
Iter 0: converting...
0
Iter 1: creating z...
tensor([[-1.2650, -1.4899,  0.8124,  ..., -0.6724, -0.9882,  1.2053]],
       device='cuda:0')
Iter 1: running model...
Iter 1: converting...
1
GenerateTime 0.5812084674835205
UniqueSmiles 2
ValidSmiles 0
UniqueAndValidMols 0
NovelMols 0
AverageRings 0
SMILE C1C(=CC2C(CC(=C1)CN1CCCC1)(C1CCOC1=CC2C(C=C1N1CC2)


## **Production evaluate chunk

In [12]:
import numpy as np
import sys, os, time, pickle
from sklearn import metrics
import torch
from rdkit.Chem import AllChem as Chem



#### MODIFY THESE ###
LATENT_DIM = 1024
eval_quant = 1000
model_path="/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/code/run4_longMask_v3_genEval_default/test_on_small/vae_decoder_len50_Mol50k_test_defaultVAE_rm.pth"



class Lang:
    def __init__(self):
        self.chartoindex = {'SOS': 1, 'EOS': 0, 'C': 2, '(': 3,
                '=': 4, 'O': 5, ')': 6, '[': 7, '-': 8, ']': 9,
                'N': 10, '+': 11, '1': 12, 'P': 13, '2': 14,'3': 15,
                '4': 16, 'S': 17, '#': 18, '5': 19,'6': 20, '7': 21,
                'H': 22, 'I': 23, 'B': 24, 'F': 25, '8': 26, '9': 27
                } 
        self.indextochar = {1: 'SOS', 0: 'EOS', 2: 'C', 3: '(',
                4: '=', 5: 'O', 6: ')', 7: '[', 8: '-', 9: ']',
                10: 'N', 11: '+', 12: '1', 13: 'P', 14: '2', 15: '3',
                16: '4', 17: 'S', 18: '#', 19: '5', 20: '6', 21: '7',
                22: 'H', 23: 'I', 24: 'B', 25: 'F', 26: '8', 27: '9'
                }
        self.nchars = 28
        
    def indexesFromSMILES(self, smiles_str):
        index_list = [self.chartoindex[char] for char in smiles_str]
        index_list.append(self.chartoindex["EOS"])
        return np.array(index_list, dtype=np.uint8)
        
    def indexToSmiles(self,indices):
        '''convert list of indices into a smiles string'''
        #todo: be nice and ignore after first EOS before this check
        if 1 in indices:
            print("SOS character encountered in smile")
            return 'x'
        smiles_str = ''.join(list(map(lambda x: self.indextochar[int(x)] if x != 0.0 else 'E',indices)))
        return smiles_str.split('E')[0] #Only want values before output 'EOS' token        


def smilesToStatistics(list_of_smiles,trainsmi):
    '''Return number valid smiles, number of unique molecules, and average number of rings'''
    count_molecules = 0
    cannonical_smiles = set()
    ringcnt = 0
    novelcnt = 0
    for smiles in list_of_smiles:
        if smiles == '':
            continue
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                count_molecules += 1
                can = Chem.MolToSmiles(mol)
                if can not in cannonical_smiles:
                    #print(can)
                    cannonical_smiles.add(can)
                    r = mol.GetRingInfo()
                    ringcnt += r.NumRings()
                    if can not in trainsmi:
                        novelcnt += 1
                    
        except:
            continue
    N = len(cannonical_smiles)
    ringave = 0 if N == 0 else ringcnt/N
    return count_molecules, N, novelcnt, ringave


lang = Lang()
print('about to load')
model = torch.jit.load(model_path)
print('loaded',flush=True)
torch.manual_seed(0)
trainsmi = set()
# if args.train_pickle:
#     trainsmi = pickle.load(open(args.train_pickle,'rb'))
print("read pickle",flush=True)
start_time = time.time() 

generated_smiles = []

# with torch.no_grad():

print('Creating z...', flush=True)
z = torch.zeros((1,LATENT_DIM),device='cuda')
print('z created on cuda -> Running model...', flush=True)
_, output_smile = model(z)
torch.cuda.synchronize() #show true GPU comp. time
print(f'Done in {time.time()-start_time:.1f}s', flush=True)
print(f'Output shape: {output_smile.shape}', flush=True)

print('Converting zsmile...', flush=True)
zsmile = lang.indexToSmiles(output_smile[0])
print(f'zsmile: {zsmile}', flush=True)

print('Starting loop...', flush=True)
for i in range(eval_quant):
    # print(f'Iter {i}: creating z...', flush=True)
    z_1 = torch.normal(0, 1, size=(1, LATENT_DIM),device='cuda')
    # print(z_1)
    # print(f'Iter {i}: running model...', flush=True)
    _, output_smile2 = model(z_1)
    torch.cuda.synchronize()
    # print(f'Iter {i}: converting...', flush=True)
    generated_smiles.append(lang.indexToSmiles(output_smile2[0]))
    print(i,flush=True)
    
duration = (time.time()-start_time)
valid_smiles, unique_mols, novelcnt, ringave = smilesToStatistics(generated_smiles,trainsmi)

print(f"\n{'='*20}")
print("GenerateTime",duration)
print("UniqueSmiles", len(set(generated_smiles)))
print("ValidSmiles", valid_smiles)
print("UniqueAndValidMols", unique_mols)
print("NovelMols", novelcnt)
print("AverageRings", ringave)
print("SMILE",zsmile)

about to load
loaded
read pickle
Creating z...
z created on cuda -> Running model...
Done in 0.1s
Output shape: torch.Size([1, 50])
Converting zsmile...
zsmile: C1C(=CC2CCCCCC23CCNC(=O)C1
Starting loop...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
2

[17:51:35] SMILES Parse Error: extra close parentheses while parsing: C1C=CS(=O)(C)C)=CC=1
[17:51:35] SMILES Parse Error: check for mistakes around position 15:
[17:51:35] C1C=CS(=O)(C)C)=CC=1
[17:51:35] ~~~~~~~~~~~~~~^
[17:51:35] SMILES Parse Error: Failed parsing SMILES 'C1C=CS(=O)(C)C)=CC=1' for input: 'C1C=CS(=O)(C)C)=CC=1'
[17:51:35] SMILES Parse Error: extra open parentheses while parsing: C12NC(C)=NC(=C(C=1)NCCN(CCCCC)CC
[17:51:35] SMILES Parse Error: check for mistakes around position 12:
[17:51:35] C12NC(C)=NC(=C(C=1)NCCN(CCCCC)CC
[17:51:35] ~~~~~~~~~~~^
[17:51:35] SMILES Parse Error: Failed parsing SMILES 'C12NC(C)=NC(=C(C=1)NCCN(CCCCC)CC' for input: 'C12NC(C)=NC(=C(C=1)NCCN(CCCCC)CC'
[17:51:35] SMILES Parse Error: extra open parentheses while parsing: C(C(=O)C(=CCCCCCC)CCCCCCC
[17:51:35] SMILES Parse Error: check for mistakes around position 2:
[17:51:35] C(C(=O)C(=CCCCCCC)CCCCCCC
[17:51:35] ~^
[17:51:35] SMILES Parse Error: Failed parsing SMILES 'C(C(=O)C(=CCCCCCC)CCCCCCC' 

### (Not working) save without no_grad, reload with no_grad

<br>

### Use `torch.jit.script`

In [3]:
import numpy as np
import sys, os, time, pickle
from sklearn import metrics
import torch
from rdkit.Chem import AllChem as Chem



#### MODIFY THESE ###
LATENT_DIM = 1024
eval_quant = 1000
model_path="/net/dali/home/mscbio/til177/Github/cobb2060-2026s/project6_VAE_PubChemSMILES/code/run4_longMask_v3_genEval_default/out_v3JitScript/vae_decoder_len150_Mol5M_defaultVAE_JitScript.pth"



class Lang:
    def __init__(self):
        self.chartoindex = {'SOS': 1, 'EOS': 0, 'C': 2, '(': 3,
                '=': 4, 'O': 5, ')': 6, '[': 7, '-': 8, ']': 9,
                'N': 10, '+': 11, '1': 12, 'P': 13, '2': 14,'3': 15,
                '4': 16, 'S': 17, '#': 18, '5': 19,'6': 20, '7': 21,
                'H': 22, 'I': 23, 'B': 24, 'F': 25, '8': 26, '9': 27
                } 
        self.indextochar = {1: 'SOS', 0: 'EOS', 2: 'C', 3: '(',
                4: '=', 5: 'O', 6: ')', 7: '[', 8: '-', 9: ']',
                10: 'N', 11: '+', 12: '1', 13: 'P', 14: '2', 15: '3',
                16: '4', 17: 'S', 18: '#', 19: '5', 20: '6', 21: '7',
                22: 'H', 23: 'I', 24: 'B', 25: 'F', 26: '8', 27: '9'
                }
        self.nchars = 28
        
    def indexesFromSMILES(self, smiles_str):
        index_list = [self.chartoindex[char] for char in smiles_str]
        index_list.append(self.chartoindex["EOS"])
        return np.array(index_list, dtype=np.uint8)
        
    def indexToSmiles(self,indices):
        '''convert list of indices into a smiles string'''
        #todo: be nice and ignore after first EOS before this check
        if 1 in indices:
            print("SOS character encountered in smile")
            return 'x'
        smiles_str = ''.join(list(map(lambda x: self.indextochar[int(x)] if x != 0.0 else 'E',indices)))
        return smiles_str.split('E')[0] #Only want values before output 'EOS' token        


def smilesToStatistics(list_of_smiles,trainsmi):
    '''Return number valid smiles, number of unique molecules, and average number of rings'''
    count_molecules = 0
    cannonical_smiles = set()
    ringcnt = 0
    novelcnt = 0
    for smiles in list_of_smiles:
        if smiles == '':
            continue
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                count_molecules += 1
                can = Chem.MolToSmiles(mol)
                if can not in cannonical_smiles:
                    #print(can)
                    cannonical_smiles.add(can)
                    r = mol.GetRingInfo()
                    ringcnt += r.NumRings()
                    if can not in trainsmi:
                        novelcnt += 1
                    
        except:
            continue
    N = len(cannonical_smiles)
    ringave = 0 if N == 0 else ringcnt/N
    return count_molecules, N, novelcnt, ringave


lang = Lang()
print('about to load')
model = torch.jit.load(model_path)
print('loaded',flush=True)
torch.manual_seed(0)
trainsmi = set()
# if args.train_pickle:
#     trainsmi = pickle.load(open(args.train_pickle,'rb'))
print("read pickle",flush=True)
start_time = time.time() 

generated_smiles = []

with torch.no_grad():
    print('Creating z...', flush=True)
    z = torch.zeros((1,LATENT_DIM),device='cuda')
    print('z created on cuda -> Running model...', flush=True)
    _, output_smile = model(z)
    torch.cuda.synchronize() #show true GPU comp. time
    print(f'Done in {time.time()-start_time:.1f}s', flush=True)
    print(f'Output shape: {output_smile.shape}', flush=True)
    
    print('Converting zsmile...', flush=True)
    zsmile = lang.indexToSmiles(output_smile[0])
    print(f'zsmile: {zsmile}', flush=True)
    
    print('Starting loop...', flush=True)
    for i in range(eval_quant):
        # print(f'Iter {i}: creating z...', flush=True)
        z_1 = torch.normal(0, 1, size=(1, LATENT_DIM),device='cuda')
        # print(z_1)
        # print(f'Iter {i}: running model...', flush=True)
        _, output_smile2 = model(z_1)
        torch.cuda.synchronize()
        # print(f'Iter {i}: converting...', flush=True)
        generated_smiles.append(lang.indexToSmiles(output_smile2[0]))
        print(i,flush=True)
    
duration = (time.time()-start_time)
valid_smiles, unique_mols, novelcnt, ringave = smilesToStatistics(generated_smiles,trainsmi)

print(f"\n{'='*20}")
print("GenerateTime",duration)
print("UniqueSmiles", len(set(generated_smiles)))
print("ValidSmiles", valid_smiles)
print("UniqueAndValidMols", unique_mols)
print("NovelMols", novelcnt)
print("AverageRings", ringave)
print("SMILE",zsmile)

about to load
loaded
read pickle
Creating z...
z created on cuda -> Running model...
Done in 0.1s
Output shape: torch.Size([1, 150])
Converting zsmile...
zsmile: C1C(=CC(OC)=C(C=1)NC(=O)C1C=CC(NC(=O)C2C=CC(=CC=2)C)=CC=1)C
Starting loop...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
SOS character encountered in smile
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208


[21:26:19] SMILES Parse Error: extra close parentheses while parsing: FC1C(C)C)NC2=CC=C(NC(=O)CCN3C(C(=O)N(C)CC3=CC(=C(OC)C=C3)OC)C)=CC(=CC=1)F
[21:26:19] SMILES Parse Error: check for mistakes around position 9:
[21:26:19] FC1C(C)C)NC2=CC=C(NC(=O)CCN3C(C(=O)N(C)CC
[21:26:19] ~~~~~~~~^
[21:26:19] SMILES Parse Error: Failed parsing SMILES 'FC1C(C)C)NC2=CC=C(NC(=O)CCN3C(C(=O)N(C)CC3=CC(=C(OC)C=C3)OC)C)=CC(=CC=1)F' for input: 'FC1C(C)C)NC2=CC=C(NC(=O)CCN3C(C(=O)N(C)CC3=CC(=C(OC)C=C3)OC)C)=CC(=CC=1)F'
[21:26:19] SMILES Parse Error: extra close parentheses while parsing: C1(NC(=O)N2CNC[NH+]3CCC4C(=CC=CC=4C)C(C)C3=O)=CC=CC=2)=CC=CC=C1
[21:26:19] SMILES Parse Error: check for mistakes around position 54:
[21:26:19] 4C)C(C)C3=O)=CC=CC=2)=CC=CC=C1
[21:26:19] ~~~~~~~~~~~~~~~~~~~~^
[21:26:19] SMILES Parse Error: Failed parsing SMILES 'C1(NC(=O)N2CNC[NH+]3CCC4C(=CC=CC=4C)C(C)C3=O)=CC=CC=2)=CC=CC=C1' for input: 'C1(NC(=O)N2CNC[NH+]3CCC4C(=CC=CC=4C)C(C)C3=O)=CC=CC=2)=CC=CC=C1'
[21:26:19] SMILES Pars